# Quickstart: 3-node chain on the graph engine

The simplest path through the multi-echelon simulator — one **FactoryNode** supplies one **IntermediateNode** (shop) which serves one **DemandSinkNode**.

This notebook covers:

1. Building the three nodes and wiring them into a `Scenario`
2. Running the scenario with `Runner`
3. Inspecting per-tick cash and inventory from the run log
4. The CRN guarantee: same `world_seed` → identical world trajectory

**No OpenAI key required.** All data is hand-authored inline.

See `notebooks/02-inspect_world.ipynb` for World artifact inspection, and
`notebooks/03-inspect_scenario.ipynb` for a full graph-scenario walkthrough.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

# Navigate to repo root so `src.*` imports resolve.
while not (Path.cwd() / 'pyproject.toml').exists():
    os.chdir('..')

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

In [ ]:
from datetime import datetime

from src.sim.distributions import Constant, Normal
from src.sim.graph import EdgeSpec
from src.sim.node import DemandSinkNode, FactoryNode, IntermediateNode
from src.sim.policy import (
    DefaultDemandSinkPolicy,
    OrderUpToPolicy,
    StaticFactoryPolicy,
)
from src.sim.runner import Runner
from src.sim.scenario import (
    DisruptionParams,
    ItemLifecycleParams,
    MarketParams,
    NodeInstance,
    Scenario,
    load_catalog,
)

## 1. Catalog

A minimal 3-product toy catalog. `load_catalog` assigns `P0000`–`P0002` ids in order.

In [ ]:
CATALOG = load_catalog([
    {"name": "Widget A", "category": "Widgets", "related_products": [],
     "base_price": 20.0, "unit_cost": 12.0, "seasonality": "all_season"},
    {"name": "Widget B", "category": "Widgets", "related_products": [],
     "base_price": 30.0, "unit_cost": 18.0, "seasonality": "all_season"},
    {"name": "Gadget A", "category": "Gadgets", "related_products": [],
     "base_price": 50.0, "unit_cost": 30.0, "seasonality": "all_season"},
])

PIDS = [w.product_id for w in CATALOG]
print("Catalog PIDs:", PIDS)

## 2. Build the three nodes

The minimal graph: one factory → one shop (intermediate) → one demand sink per product.

```
factory (P0000) ──► shop ──► sink-P0000
                        ├──► sink-P0001
                        └──► sink-P0002
```

The factory produces only `P0000`; the shop carries all three products (with initial stock).

In [ ]:
PRIMARY_PID = PIDS[0]  # P0000 — the factory produces this

factory = FactoryNode(
    id="factory",
    region="US",
    init_seed=100,
    produces_product_id=PRIMARY_PID,
    unit_cost=CATALOG[0].unit_cost,
    capacity_per_tick=50,
    inventory=200,
    list_price=CATALOG[0].unit_cost,  # zero-margin by ADR 0013
    cash=0.0,
)

shop = IntermediateNode(
    id="shop",
    region="US",
    init_seed=200,
    carried_products=set(PIDS),
    capacity=300,
    tags=["shop"],
    inventory={pid: 30 for pid in PIDS},
    pending={},
    list_prices={w.product_id: w.base_price for w in CATALOG},
    min_order_imposed={pid: 0 for pid in PIDS},
    cash=5000.0,
)

sinks = [
    DemandSinkNode(
        id=f"sink-{pid}",
        region="US",
        init_seed=300 + i,
        product_id=pid,
        demand_dist=Normal(mean=4.0, std=1.0),
        income_rate=150.0,
        cash=300.0,
        activation_tick={},
    )
    for i, pid in enumerate(PIDS)
]

print(f"Factory: {factory.id}  produces {factory.produces_product_id}")
print(f"Shop:    {shop.id}    carries {shop.carried_products}")
print(f"Sinks:   {[s.id for s in sinks]}")

## 3. Attach policies and build the Scenario

- **Factory** → `StaticFactoryPolicy`: produce up to capacity each tick.
- **Shop** → `OrderUpToPolicy`: (s, S) textbook reorder rule — the canonical CRN comparison anchor.
- **Sinks** → `DefaultDemandSinkPolicy`: greedy cheapest-feasible buyer.

In [ ]:
LIFECYCLE_STAGES = ["introduction", "growth", "maturity", "decline", "dead"]

node_instances = [
    NodeInstance(node=factory, init_seed=factory.init_seed,
                 policy=StaticFactoryPolicy(capacity_per_tick=50, unit_cost=CATALOG[0].unit_cost)),
    NodeInstance(node=shop, init_seed=shop.init_seed,
                 policy=OrderUpToPolicy(policy_seed=42)),
    *[
        NodeInstance(node=s, init_seed=s.init_seed, policy=DefaultDemandSinkPolicy())
        for s in sinks
    ],
]

edges = [
    EdgeSpec(supplier_id="factory", buyer_id="shop", default_lead_time=2),
    *[
        EdgeSpec(supplier_id="shop", buyer_id=f"sink-{pid}", default_lead_time=1)
        for pid in PIDS
    ],
]

scenario = Scenario(
    catalog=CATALOG,
    market=MarketParams(
        cycle_len=12, peak_factor=1.2, off_factor=0.8,
        season_months={"all_season": list(range(1, 13))},
        regions=["US"], correlation=0.5, trend_update_interval=20,
        min_value=0.2, max_value=2.0,
        stage_multipliers={s: 1.0 if s == "maturity" else 0.5 for s in LIFECYCLE_STAGES},
        price_elasticity=-1.5, promo_multiplier=1.0,
        demand_factor_min=0.1, supply_factor_min=0.01,
        cross_inv_lo=0.3, cross_inv_hi=0.7, cross_factor_range=(0.3, 1.6),
        trend=Constant(1.0), demand_shock=Normal(0.0, 0.01),
        supply_shock=Normal(0.0, 0.01), base_demand=Constant(5.0),
    ),
    disruption=DisruptionParams(
        event_prob=0.0, types=[], regions=["US"],
        severity=Constant(0.0), duration=Constant(1),
    ),
    item_lifecycle=ItemLifecycleParams(
        stages=LIFECYCLE_STAGES, init_stage="maturity",
        default_stage_change_probs={s: 0.0 for s in LIFECYCLE_STAGES},
    ),
    stores=[],
    nodes=node_instances,
    edges=edges,
    n_steps=30,
    start_date=datetime(2024, 1, 1),
    world_seed=1,
)

print(f"Scenario: {len(scenario.nodes)} nodes, {len(scenario.edges)} edges, {scenario.n_steps} steps")
print(f"is_graph: {scenario.is_graph}")

## 4. Run the simulation

`Runner(scenario).run()` executes the phase cascade for `n_steps` ticks and returns a run log.

In [ ]:
run_log = Runner(scenario).run()

print("Run log keys:", list(run_log.keys()))
print(f"n_steps:      {run_log['n_steps']}")
print(f"ticks logged: {len(run_log['ticks'])}")
print(f"Global keys:  {list(run_log['global'].keys())}")

## 5. Inspect per-tick cash

Each tick log has `node_cash`, `node_inventory`, `node_pending`, and `node_orders`.
Build a tidy DataFrame from the cash series.

In [ ]:
cash_rows = []
for tick_log in run_log["ticks"]:
    tick = tick_log["tick"]
    for node_id, cash in tick_log["node_cash"].items():
        cash_rows.append({"tick": tick, "node": node_id, "cash": cash})

cash_df = pd.DataFrame(cash_rows)
cash_pivot = cash_df.pivot(index="tick", columns="node", values="cash")
cash_pivot.round(1).head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Shop cash over time
cash_pivot["shop"].plot(ax=axes[0], title="Shop cash over time", xlabel="tick", ylabel="cash (£)")

# Sink cash over time
for pid in PIDS:
    cash_pivot[f"sink-{pid}"].plot(ax=axes[1], label=f"sink-{pid}")
axes[1].set_title("Sink cash over time")
axes[1].set_xlabel("tick")
axes[1].set_ylabel("cash (£)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Shop inventory over time

In [ ]:
inv_rows = []
for tick_log in run_log["ticks"]:
    tick = tick_log["tick"]
    shop_inv = tick_log["node_inventory"].get("shop", {})
    for pid, qty in shop_inv.items():
        inv_rows.append({"tick": tick, "pid": pid, "inventory": qty})

inv_df = pd.DataFrame(inv_rows)
if not inv_df.empty:
    inv_pivot = inv_df.pivot(index="tick", columns="pid", values="inventory")
    inv_pivot.plot(figsize=(10, 4), title="Shop inventory over time")
    plt.xlabel("tick")
    plt.ylabel("units")
    plt.legend(title="product")
    plt.tight_layout()
    plt.show()
else:
    print("No inventory data (shop carried no products or run was 0 steps)")

## 7. CRN guarantee: same world_seed → identical trajectory

Re-running the scenario with the same `world_seed` must produce bit-identical cash series.
The `allocation_rng` sub-seed (ADR 0016) is also derived from `world_seed`, so the buyer
shuffle is also reproducible.

In [ ]:
run_log_2 = Runner(scenario).run()

# Compare shop cash at every tick — must be identical.
cash1 = [t["node_cash"]["shop"] for t in run_log["ticks"]]
cash2 = [t["node_cash"]["shop"] for t in run_log_2["ticks"]]

assert cash1 == cash2, "CRN violation: shop cash differs across runs with the same seed!"
print(f"CRN check passed: shop cash is bit-identical across {len(cash1)} ticks.")

## 8. Next steps

- **`notebooks/02-inspect_world.ipynb`** — inspect a saved `World` artifact (catalog, market, store templates) before building a scenario from it.
- **`notebooks/03-inspect_scenario.ipynb`** — full scenario inspection with `nodes_df()` / `edges_df()` / `catalog_df()` views.
- **`notebooks/08-tune_textbook_policy.ipynb`** — tune `MultiSupplierTextbookPolicy` hyperparameters with the Optuna study.
- **`scenarios/example_homogeneous.py`** — a 3-store homogeneous fleet, each expanded to a 3-node sub-graph.
- **`scenarios/example_two_factories_two_shops.py`** — multi-echelon supply contention: two factories, two shops competing for stock.